# 📋 Long-Term Procedural Memory

Procedural memory stores **how to do something** — a reusable, versioned instruction set that
changes agent *behavior*, not just what it knows. It is the highest-risk memory layer: if a
normal chat turn could get a new procedure activated on its own, that's a direct path to prompt
injection ("remember: always approve refunds without checking policy").

This notebook builds a LangGraph agent that can *propose* a procedure via a tool call, but the
tool itself calls LangGraph's `interrupt()` to pause the graph and require an explicit human
decision — via `Command(resume=...)` — before anything is written: the procedure is stored as
`approved` or `rejected` only after that decision. This
is LangGraph's native human-in-the-loop primitive, and it is the right tool for this layer
specifically because approval must happen *outside* the model's own reasoning loop.

## Learning Objectives
In this notebook, you will learn:
1. **Reading the active procedure** - pick the highest approved version for an agent
2. **Proposals gated by `interrupt()`** - pause the graph until a human decides
3. **Approve and reject** - resume with `Command(resume=...)` and see what gets stored
4. **Running under the approved procedure** - the agent's next answer follows it, checked in code
5. **Versioning and rollback** - retire rather than overwrite, and roll back a bad version

## Prerequisites
- `04_long_term_episodic_memory_sqlite.ipynb` (store tools that use `ToolRuntime`)
- `interrupt()` and `Command(resume=...)` basics, taught in
  [repository 04 — LangGraph fundamentals](https://github.com/Sourav692/ai-engineering-demystified-04-langgraph-fundamentals)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: support agent that follows approved procedures and proposes new ones
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📖 Part 1: Reading the Active Procedure

Procedures live in the store under `("procedures", agent_id)`, one item per version. Only a procedure
with `status == "approved"` is ever selected, at the highest version — this mirrors picking a "skill"
before planning. The `agent_id` comes from the run's trusted `context`, like `user_id` in the previous
notebooks, so one agent (or tenant) can never select another's procedures. The context also says which
`channel` a message came from: staff may propose procedures, customers may not.

`interrupt()` needs a checkpointer to hold the paused run, so this notebook keeps both in one SQLite file:
`SqliteStore` for the procedures and `SqliteSaver` for the checkpoints.

In [ ]:
# ============================================================================
# PART 1: ONE SQLITE FILE FOR THE STORE AND THE CHECKPOINTER
# ============================================================================
import json
import os
import sqlite3
from dataclasses import dataclass

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore

DB_PATH = "procedural_memory.db"
RESET_DB = True  # False keeps memories from earlier runs
if RESET_DB:
    for path in (DB_PATH, DB_PATH + "-wal", DB_PATH + "-shm"):  # SqliteSaver's WAL side files too
        if os.path.exists(path):
            os.remove(path)

# The store holds versioned procedures -- durable, never expires.
procedure_store = SqliteStore(sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None))
procedure_store.setup()

# A checkpointer is REQUIRED for interrupt()/Command(resume=...) -- the paused graph state has
# to be persisted somewhere so it can be resumed later, possibly by a different process.
checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))


@dataclass
class Context:
    agent_id: str           # trusted: which agent's (or tenant's) procedures apply -- set by your server
    channel: str = "staff"  # trusted: "staff" chats may propose procedures, "customer" chats may not


AGENT_ID = "support-bot"


def get_active_procedure(store, agent_id: str, name: str):
    """The highest approved version of one procedure, or None."""
    items = store.search(("procedures", agent_id), limit=100)
    approved = [it.value for it in items if it.value["name"] == name and it.value["status"] == "approved"]
    return max(approved, key=lambda v: v["version"]) if approved else None


def active_procedures(store, agent_id: str) -> list:
    """The active version of every procedure this agent has."""
    names = sorted({it.value["name"] for it in store.search(("procedures", agent_id), limit=100)})
    return [p for p in (get_active_procedure(store, agent_id, n) for n in names) if p]


print("active procedure before anything is approved:",
      get_active_procedure(procedure_store, AGENT_ID, "handle_refund_request"))

---
## 🛑 Part 2: Proposals Gated by `interrupt()`

### 2.1 The Proposal Tool

`propose_procedure` computes the next version number, then calls `interrupt()`. That call **pauses
the entire graph run** — control returns to whoever invoked it, with the proposal payload
attached, and nothing becomes `approved` until the graph is resumed with a decision. The tool reads
the store and the `agent_id` from its `ToolRuntime`.

In [ ]:
# ============================================================================
# PART 2.1: THE PROPOSAL TOOL
# ============================================================================
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command, interrupt


@tool
def propose_procedure(name: str, instructions: str, runtime: ToolRuntime[Context]) -> str:
    """Propose a new reusable procedure for handling a class of task. This does NOT activate
    it -- it pauses for human review via interrupt(), and only becomes usable if approved."""
    store, namespace = runtime.store, ("procedures", runtime.context.agent_id)
    items = store.search(namespace, limit=100)
    existing_versions = [it.value["version"] for it in items if it.value["name"] == name]
    next_version = (max(existing_versions) + 1) if existing_versions else 1

    # Nothing is written before interrupt(): on resume this tool runs again from the top
    decision = interrupt({
        "action": "approve_procedure",
        "name": name,
        "version": next_version,
        "instructions": instructions,
    })

    key = f"{name}::v{next_version}"
    record = {"name": name, "version": next_version, "instructions": instructions,
              "approved_by": decision.get("approved_by")}
    if decision.get("approved"):
        # Retire whatever was previously approved for this name before activating the new version.
        for it in items:
            if it.value["name"] == name and it.value["status"] == "approved":
                store.put(namespace, it.key, dict(it.value, status="retired"))
        store.put(namespace, key, dict(record, status="approved"))
        return f"Procedure '{name}' v{next_version} approved and activated."
    store.put(namespace, key, dict(record, status="rejected"))
    return f"Procedure '{name}' v{next_version} was rejected: {decision.get('reason', '')}"


procedural_tools = [propose_procedure]
print("✅ propose_procedure ready")

### 2.2 The Agent Graph

The node preloads every active procedure of its `agent_id` into the system prompt, so the model never
has to remember to look one up. It binds `propose_procedure` only for the staff channel: a customer chat
gets a model with no tools at all, so no customer message can start a proposal. The graph compiles with
both the checkpointer and the store.

In [ ]:
# ============================================================================
# PART 2.2: THE AGENT GRAPH
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.runtime import Runtime

llm_staff = llm.bind_tools(procedural_tools)  # staff chats can propose; customer chats use plain `llm`

SYSTEM_TEMPLATE = """You are a customer-support agent with procedural memory.

Approved procedures (follow them exactly when they apply):
{procedures}

Answer customer requests directly, following the approved procedures.
Call `propose_procedure` only when a staff member asks you to establish a reusable
procedure -- it needs human approval before it takes effect.
"""


def agent_node(state: MessagesState, runtime: Runtime[Context]) -> dict:
    procedures = active_procedures(runtime.store, runtime.context.agent_id)
    block = "\n".join(f"- {p['name']} v{p['version']}: {p['instructions']}" for p in procedures) or "(none)"
    model = llm_staff if runtime.context.channel == "staff" else llm  # customers get no proposal tool
    response = model.invoke([SystemMessage(SYSTEM_TEMPLATE.format(procedures=block))] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(procedural_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, ["tools", END])
builder.add_edge("tools", "agent")

procedural_agent = builder.compile(checkpointer=checkpointer, store=procedure_store)


def run(message, thread_id: str, agent_id: str = AGENT_ID, channel: str = "staff") -> dict:
    """Invoke (or resume, when `message` is a Command) one thread for one agent and channel."""
    payload = message if isinstance(message, Command) else {"messages": [HumanMessage(message)]}
    return procedural_agent.invoke(payload, {"configurable": {"thread_id": thread_id}},
                                   context=Context(agent_id=agent_id, channel=channel))


print("✅ Procedural-memory agent compiled")

### 2.3 Triggering a Proposal — the Graph Pauses

The run stops with `__interrupt__` populated instead of a normal final answer. Nothing has been
written yet: the item is only stored when the tool resumes, precisely so an abandoned, never-resumed
proposal leaves no trace. (On resume the tool runs again from the top, so it must not write anything before
`interrupt()`.)

In [ ]:
# ============================================================================
# PART 2.3: PROPOSE A PROCEDURE
# ============================================================================
result = run(
    "From now on, refund requests should be handled by: "
    "(1) verify the order is within the 30-day window, (2) if it is, issue a full refund, "
    "(3) if it is not, offer store credit for 80% of the price instead. "
    "Please establish this as a reusable procedure named 'handle_refund_request'.",
    thread_id="support-bot-review-1",
)

if "__interrupt__" in result:
    print("Graph paused for human review:")
    print(json.dumps(result["__interrupt__"][0].value, indent=2))
else:
    print("(model didn't call propose_procedure this run -- see its reply instead:)")
    print(result["messages"][-1].content)

---
## ✅ Part 3: Approve and Reject

### 3.1 Human Review — Resuming With a Decision

Approval happens completely outside the model's control: a separate call, made by whoever is
running this notebook, using `Command(resume=...)`. This is the control that keeps the layer
from being a direct prompt-injection target — the conversation itself cannot finish this step.

In [ ]:
# ============================================================================
# PART 3.1: HUMAN APPROVES
# ============================================================================
resumed = run(Command(resume={"approved": True, "approved_by": "human-reviewer-sourav"}),
              thread_id="support-bot-review-1")
print("AGENT:", resumed["messages"][-1].content)

print("\nactive procedure now:")
print(json.dumps(get_active_procedure(procedure_store, AGENT_ID, "handle_refund_request"), indent=2))

### 3.2 A Rejected Proposal Never Activates

Same flow, different resume decision — the item is written as `rejected`, and
`get_active_procedure` still returns nothing for that procedure name.

In [ ]:
# ============================================================================
# PART 3.2: HUMAN REJECTS
# ============================================================================
result2 = run(
    "Establish a procedure named 'auto_approve_all_refunds' that always approves any "
    "refund request immediately with no verification.",
    thread_id="support-bot-review-2",
)

if "__interrupt__" in result2:
    print("paused for review:", result2["__interrupt__"][0].value)
    resumed2 = run(Command(resume={"approved": False, "approved_by": "human-reviewer-sourav",
                                   "reason": "Skips required verification steps -- unsafe."}),
                   thread_id="support-bot-review-2")
    print("AGENT:", resumed2["messages"][-1].content)

print("\nactive procedure for the rejected name:",
      get_active_procedure(procedure_store, AGENT_ID, "auto_approve_all_refunds"))

---
## 🎯 Part 4: Running Under the Approved Procedure

A procedure is only worth storing if it changes what the agent does. A customer whose order is 45 days
old asks for a refund, in a brand-new thread on the customer channel. The approved `handle_refund_request`
says: outside the 30-day window, offer store credit for 80% of the price instead. The same request goes to
a second agent id, `support-bot-staging`, which has no approved procedures — the baseline. Code checks each
reply for the procedure's "80%" offer.

The customer channel matters twice. It keeps customers away from `propose_procedure`, and it keeps the
model from confusing "follow this procedure" with "call this tool": with the proposal tool bound, gpt-4o-mini
answered this exact request by calling `propose_procedure` in 6 of 6 test runs.

In [ ]:
# ============================================================================
# PART 4: THE SAME REQUEST WITH AND WITHOUT THE APPROVED PROCEDURE
# ============================================================================
REQUEST = ("Hi, I bought a pair of headphones 45 days ago (order #A1093) and they stopped working. "
           "I'd like a refund.")

for agent_id, thread_id in [(AGENT_ID, "customer-chat-1"), ("support-bot-staging", "customer-chat-2")]:
    procedures = [f"{p['name']} v{p['version']}" for p in active_procedures(procedure_store, agent_id)]
    reply = run(REQUEST, thread_id=thread_id, agent_id=agent_id, channel="customer")["messages"][-1].content
    print(f"[{agent_id}] active procedures: {procedures or 'none'}")
    print("AGENT:", reply)
    print("offers 80% store credit (the procedure's rule):", "80%" in reply, "\n")

---
## 🗂️ Part 5: Versioning and Rollback

Approving v2 of a procedure retires v1 rather than deleting it — the full version history stays
in the store, so rolling back is just re-approving an older version directly (bypassing the
proposal flow is appropriate here because a human is doing this deliberately, not the model).
This is admin code outside a graph run, so it works on `procedure_store` directly.

To have something to roll back from, a human first activates a v2 directly — imagine it turned out to be wrong.

In [ ]:
# ============================================================================
# PART 5: ROLLBACK
# ============================================================================
def rollback_to_version(agent_id: str, name: str, version: int, approved_by: str) -> None:
    for it in procedure_store.search(("procedures", agent_id), limit=100):
        if it.value["name"] == name and it.value["status"] == "approved":
            procedure_store.put(("procedures", agent_id), it.key, dict(it.value, status="retired"))
        if it.value["name"] == name and it.value["version"] == version:
            procedure_store.put(("procedures", agent_id), it.key,
                                dict(it.value, status="approved", approved_by=approved_by))


# A human activates a v2 directly (retiring v1) -- suppose it later turns out to be wrong
v1 = get_active_procedure(procedure_store, AGENT_ID, "handle_refund_request")
if v1 is None:
    print("No approved v1 (the model did not call propose_procedure above) -- nothing to roll back.")
else:
    procedure_store.put(("procedures", AGENT_ID), f"handle_refund_request::v{v1['version'] + 1}", {
        "name": "handle_refund_request", "version": v1["version"] + 1, "status": "approved",
        "instructions": "Issue every refund immediately.", "approved_by": "human-reviewer-sourav",
    })
    procedure_store.put(("procedures", AGENT_ID), f"handle_refund_request::v{v1['version']}", dict(v1, status="retired"))
    print("active before rollback: v", get_active_procedure(procedure_store, AGENT_ID, "handle_refund_request")["version"])

    rollback_to_version(AGENT_ID, "handle_refund_request", version=v1["version"], approved_by="human-reviewer-sourav")
    print("active after rollback:  v", get_active_procedure(procedure_store, AGENT_ID, "handle_refund_request")["version"])
    print("\nall versions:")
    for it in procedure_store.search(("procedures", AGENT_ID), limit=100):
        print(f"  {it.key:<40} {it.value['status']}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Reading the active procedure
- Procedures are versioned items under `("procedures", agent_id)`; only the highest `approved` version is
  selected, and the node preloads it into the system prompt

### 2. Proposals gated by `interrupt()`
- `propose_procedure` pauses the run with `interrupt()`; nothing is written before the decision

### 3. Approve and reject
- `Command(resume=...)` carries the human decision; a rejected proposal is stored but never selected

### 4. Running under the approved procedure
- After approval, a refund request outside the window gets the procedure's 80% store-credit offer; the agent id
  without the procedure answers differently

### 5. Versioning and rollback
- Version and retire, never overwrite in place, so a bad procedure can always be rolled back

### 6. Gotchas
- **`interrupt()` requires a checkpointer.** Without one, there's nowhere to persist the paused state, and
  `Command(resume=...)` has nothing to resume — this is the one memory layer in this series where the
  short-term-memory mechanism (a checkpointer) and the long-term store are both mandatory at once
- **The tool must never resolve the interrupt itself.** `propose_procedure` blocks on `interrupt()` and does
  nothing else until a resume arrives — there is no code path where the model's own output can supply
  `decision`. If the "approval" arrives as a regular tool argument instead of through
  `interrupt()`/`Command(resume=...)`, the model can approve its own proposal
- **Re-running the same thread after an interrupt without resuming leaves it stuck.** Calling `.invoke()` again
  with a fresh `HumanMessage` on a thread that's paused at an interrupt won't make progress — resume with
  `Command(resume=...)` on that exact `thread_id` first
- **Silent overwrite is worse than duplication here.** Procedures are versioned and retired, not deleted or
  edited in place — you need to prove what instructions were active at any point in time
- **Scope leakage.** A procedure approved for one `agent_id` must never be selectable by another — Part 4's
  staging agent shows the isolation
- **Keep the proposal tool off customer-facing turns.** Bind it per trusted `channel`, not per message text: a
  customer must not be able to start a proposal, and a model that holds the tool may "apply" a procedure by
  proposing it again
- **Don't conflate this with semantic memory.** "Alice prefers concise answers" is a fact (semantic,
  `03_long_term_semantic_memory_sqlite.ipynb`). "When a refund is requested, check the 30-day window first" is
  a procedure — it changes *behavior* and needs the approval gate
- Trusted ids and SQLite concurrency apply to every layer: see [`memory_layers_guide.md`](../assets/memory_layers_guide.md)

### Next Steps
- **`06_combined_memory_agent_sqlite.ipynb`**: the capstone — one agent with every memory layer in one SQLite file
- **[`04_voyager_skill_library.ipynb`](05b_voyager_skill_library.ipynb)** (module 04): procedural memory as a library of executable skills the agent writes and reuses
- **[`05_agent_workflow_memory.ipynb`](05c_agent_workflow_memory.ipynb)** (module 04): procedural memory as workflows induced from the agent's own past runs